# what is this notebook

Downloads the HotpotQA dataset

In [1]:
%pip install datasets pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 980.8 kB/s eta 0:00:00 0:00:01
  Using cached filelock-3.17.0-py3-none-any.whl.metadata (2.9 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 646.2 kB/s eta 0:00:000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 368.9 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 236.3 kB/s eta 0:00:00a 0:00:01
  Using cached typing_extensions-4.12.2-py3-none-any.whl.metadata (3.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 94.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 222.7 kB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 126.9 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 158.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 209.5 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.0/46

In [2]:
from datasets import load_dataset

# Load full dataset (10x larger than normal)
dataset = load_dataset("hotpot_qa", "fullwiki")

# Or load the distractor version (smaller)
# dataset = load_dataset("hotpot_qa", "distractor")

/home/zzuo_umass_edu/groupdir/zzuo/AgenticRAG/Langgraph_ARAG_Experiment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sqlite3
from datasets import load_dataset
import pandas as pd

# Load dataset
dataset = load_dataset("hotpot_qa", "fullwiki")  # or "distractor"

# Create SQLite database
conn = sqlite3.connect('hotpotqa.db')
cursor = conn.cursor()

# Create table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS hotpotqa (
        id INTEGER PRIMARY KEY,
        question TEXT,
        context TEXT,
        answer TEXT,
        supporting_facts TEXT,
        split TEXT
    )
''')

# Insert data
for split in ['train', 'validation']:
    for row in dataset[split]:
        cursor.execute('''
            INSERT INTO hotpotqa 
            (question, context, answer, supporting_facts, split)
            VALUES (?, ?, ?, ?, ?)
        ''', (
            row['question'],
            str(row['context']),  # Convert list of contexts to string
            row['answer'],
            str(row['supporting_facts']),
            split
        ))

conn.commit()
conn.close()